In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
import numpy as np
import cv2

In [2]:
import os
import cv2
import numpy as np

In [ ]:
#download dataset from https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset

['test_set', 'training_set']

In [3]:
train_gen = ImageDataGenerator(rescale=1./255)
test_gen  = ImageDataGenerator(rescale=1./255)

In [4]:

train_data = train_gen.flow_from_directory(
    "catsanddogs/training_set/training_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
    
)

test_data = test_gen.flow_from_directory(
    "catsanddogs/test_set/test_set/",
    target_size=(128, 128),
    batch_size=32,
    class_mode='categorical'
)

Found 8006 images belonging to 2 classes.
Found 2023 images belonging to 2 classes.


In [12]:
print({v:k for k,v in test_data.class_indices.items()}[1])

dogs


In [13]:
label_map = train_data.class_indices
inv_label_map = {v: k for k, v in label_map.items()}
print("Label mapping:", inv_label_map)

Label mapping: {0: 'cats', 1: 'dogs'}


In [14]:
model =  Sequential()
model.add(Conv2D(32, (3,3), activation='relu', input_shape = (128,128,3)))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Conv2D(64, (3,3), activation='relu'))

C:\Users\memon\anaconda3\envs\vision\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 56,320 (220.00 KB)

 Trainable params: 56,320 (220.00 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.add(Flatten())
model.add(Dense(64,activation='relu'))
model.add(Dense(32,activation='relu'))
model.add(Dense(16,activation='relu'))
model.add(Dense(2,activation='softmax'))

In [17]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 50176)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     3,211,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,270,290 (12.48 MB)

 Trainable params: 3,270,290 (12.48 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [19]:
history = model.fit(
    train_data,
    epochs=10,
    validation_data=test_data
)

Epoch 1/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 255s 1s/step - accuracy: 0.4953 - loss: 0.6952 - val_accuracy: 0.5002 - val_loss: 0.7054
Epoch 2/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 114s 453ms/step - accuracy: 0.5525 - loss: 0.6878 - val_accuracy: 0.5764 - val_loss: 0.6790
Epoch 3/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 109s 435ms/step - accuracy: 0.5751 - loss: 0.6714 - val_accuracy: 0.6055 - val_loss: 0.6559
Epoch 4/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 115s 455ms/step - accuracy: 0.6279 - loss: 0.6326 - val_accuracy: 0.6653 - val_loss: 0.5971
Epoch 5/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 113s 449ms/step - accuracy: 0.6979 - loss: 0.5735 - val_accuracy: 0.6985 - val_loss: 0.5646
Epoch 6/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 104s 413ms/step - accuracy: 0.7526 - loss: 0.5045 - val_accuracy: 0.7336 - val_loss: 0.5220
Epoch 7/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 109s 435ms/step - accuracy: 0.7864 - loss: 0.4460 - val_accuracy: 0.7370 - val_loss: 0.5294
Epoch 8/10
251/251 ━━━━━━━━━━━━━━━━━━━━ 105s 418ms/step - accuracy: 0.8331 - lo

In [20]:
model.save("cat_dog_cnn_model.keras")
print("Model Saved Successfully!")

Model Saved Successfully!


In [21]:
model = load_model("cat_dog_cnn_model.keras")
print("Model Loaded Successfully!")

Model Loaded Successfully!


In [22]:
def predict_image(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (128, 128))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    preds = model.predict(img)
    index = np.argmax(preds)          # no if/else
    label = inv_label_map[index]      # label from mapping

    print("\nPrediction Scores:", preds)
    print("Predicted Label:", label)

In [23]:
predict_image("dog2.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 148ms/step

Prediction Scores: [[0.04911759 0.95088243]]
Predicted Label: dogs
